## PS2 - campus route search (Greedy Best-First vs A*)

weighted bidirectional graph, each node has a heuristic h(n) estimating cost to the goal. Greedy uses f(n)=h(n), A* uses f(n)=g(n)+h(n).

In [1]:
graph = {
    'A':[('B',4),('C',2)], 'B':[('A',4),('D',5)],
    'C':[('A',2),('D',3),('E',6)], 'D':[('B',5),('C',3),('E',3)],
    'E':[('C',6),('D',3),('F',2)], 'F':[('E',2)]
}
h = {'A':7,'B':8,'C':5,'D':4,'E':2,'F':0}
print(graph)
print(h)

{'A': [('B', 4), ('C', 2)], 'B': [('A', 4), ('D', 5)], 'C': [('A', 2), ('D', 3), ('E', 6)], 'D': [('B', 5), ('C', 3), ('E', 3)], 'E': [('C', 6), ('D', 3), ('F', 2)], 'F': [('E', 2)]}
{'A': 7, 'B': 8, 'C': 5, 'D': 4, 'E': 2, 'F': 0}


hand check first. from A, neighbors are B(h=8) and C(h=5). greedy always jumps to whichever neighbor has the smallest h, so it goes A -> C. from C, neighbors are D(h=4) and E(h=2), and E is way lower, so greedy should go to E next, not D. let's see if the code agrees.

In [2]:
import heapq, time

def path_cost(graph, path):
    total = 0
    for i in range(len(path)-1):
        for nb, c in graph[path[i]]:
            if nb == path[i+1]:
                total += c
                break
    return total

def greedy_best_first(graph, h, start, goal):
    closed = set()
    parent = {}
    frontier = [(h[start], start)]
    nodes_expanded = 0
    while frontier:
        _, cur = heapq.heappop(frontier)
        if cur in closed:
            continue
        closed.add(cur)
        nodes_expanded += 1
        if cur == goal:
            path = [cur]
            while path[-1] != start:
                path.append(parent[path[-1]])
            path.reverse()
            return True, path, path_cost(graph, path), nodes_expanded
        for nb, cost in graph.get(cur, []):
            if nb not in closed:
                if nb not in parent:
                    parent[nb] = cur
                heapq.heappush(frontier, (h[nb], nb))
    return False, [], 0, nodes_expanded

print(greedy_best_first(graph, h, 'A', 'F'))

(True, ['A', 'C', 'E', 'F'], 10, 4)


so greedy actually goes A -> C -> E -> F, cost 10. that matches the hand check (E beats D on h), but it does NOT match the path printed in the assignment PDF's sample output (A -> C -> D -> E -> F). the total cost still comes out to 10 either way, purely because both routes happen to cost the same on this particular graph, so the numeric answer in the PDF is right by coincidence but the stated path for greedy isn't what a real h(n)-only search produces. going with the correct simulation, not force-matching the PDF's path.

In [3]:
def a_star(graph, h, start, goal):
    best_g = {start: 0}
    parent = {}
    frontier = [(h[start], start)]
    closed = set()
    nodes_expanded = 0
    while frontier:
        f, cur = heapq.heappop(frontier)
        if cur in closed:
            continue
        closed.add(cur)
        nodes_expanded += 1
        if cur == goal:
            path = [cur]
            while path[-1] != start:
                path.append(parent[path[-1]])
            path.reverse()
            return True, path, best_g[goal], nodes_expanded
        for nb, cost in graph.get(cur, []):
            new_g = best_g[cur] + cost
            if nb not in best_g or new_g <= best_g[nb]:
                best_g[nb] = new_g
                parent[nb] = cur
                heapq.heappush(frontier, (new_g + h[nb], nb))
    return False, [], 0, nodes_expanded

print(a_star(graph, h, 'A', 'F'))

(True, ['A', 'C', 'D', 'E', 'F'], 10, 5)


A* gives A -> C -> D -> E -> F, cost 10, this time matching the PDF's sample output exactly (path and cost). makes sense, A* tracks real distance travelled (g) plus the heuristic, so it doesn't get fooled by a node that just looks close on paper. tie-breaking detail: A-C-D-E-F and A-C-E-F both cost 10 on this graph (2+3+3+2 = 2+6+2), so A* picking one over the other on a tie is fine, both are optimal, and this implementation's tie rule (update path if the new g is <= best known g) happens to land on the same tied path the PDF shows.

now build a second graph on purpose where greedy actually loses, per the spec's own ask for sample test case 2. idea: put a node right next to the start with a very tempting low heuristic, but a genuinely expensive edge onward, while a slightly-less-tempting node leads to a cheap route.

In [4]:
graph2 = {'S':[('A',1),('B',2)], 'A':[('S',1),('T',20)], 'B':[('S',2),('T',2)], 'T':[('A',20),('B',2)]}
h2 = {'S':3, 'A':1, 'B':2, 'T':0}
print('greedy:', greedy_best_first(graph2, h2, 'S', 'T'))
print('a star:', a_star(graph2, h2, 'S', 'T'))

greedy: (True, ['S', 'A', 'T'], 21, 3)
a star: (True, ['S', 'B', 'T'], 4, 4)


there it is. greedy sees A has h=1 (lowest looking option) and commits to it, walks S -> A -> T, total cost 21. A* isn't fooled, it knows S -> B -> T is only 4 total even though B's heuristic (2) looked worse than A's (1) at the start. this is the textbook failure mode of greedy: it only trusts the estimate to the goal, never what it already spent getting there.

finally, the full script that reads stdin and prints in the spec's format, run for real on the original sample test case.

In [5]:
import io, sys as _sys
_sys.stdin = io.StringIO('6 7\nA B 4\nA C 2\nB D 5\nC D 3\nC E 6\nD E 3\nE F 2\nA F\nA 7\nB 8\nC 5\nD 4\nE 2\nF 0')
__name__ = '__main__'
import sys
import time
import heapq


def read_input():
    data = sys.stdin.read().split("\n")
    idx = 0
    n, m = map(int, data[idx].split())
    idx += 1

    graph = {}
    for _ in range(m):
        u, v, c = data[idx].split()
        c = int(c)
        graph.setdefault(u, []).append((v, c))
        graph.setdefault(v, []).append((u, c))
        idx += 1

    start, goal = data[idx].split()
    idx += 1

    h = {}
    for _ in range(n):
        node, val = data[idx].split()
        h[node] = int(val)
        idx += 1

    return graph, h, start, goal


def path_cost(graph, path):
    total = 0
    for i in range(len(path) - 1):
        for nb, c in graph[path[i]]:
            if nb == path[i + 1]:
                total += c
                break
    return total


def greedy_best_first(graph, h, start, goal):
    start_time = time.time()
    closed = set()
    parent = {}
    frontier = [(h[start], start)]
    nodes_expanded = 0

    while frontier:
        _, cur = heapq.heappop(frontier)
        if cur in closed:
            continue
        closed.add(cur)
        nodes_expanded += 1

        if cur == goal:
            path = [cur]
            while path[-1] != start:
                path.append(parent[path[-1]])
            path.reverse()
            return True, path, path_cost(graph, path), nodes_expanded, time.time() - start_time

        for nb, cost in graph.get(cur, []):
            if nb not in closed:
                if nb not in parent:
                    parent[nb] = cur
                heapq.heappush(frontier, (h[nb], nb))

    return False, [], 0, nodes_expanded, time.time() - start_time


def a_star(graph, h, start, goal):
    start_time = time.time()
    best_g = {start: 0}
    parent = {}
    frontier = [(h[start], start)]
    closed = set()
    nodes_expanded = 0

    while frontier:
        f, cur = heapq.heappop(frontier)
        if cur in closed:
            continue
        closed.add(cur)
        nodes_expanded += 1

        if cur == goal:
            path = [cur]
            while path[-1] != start:
                path.append(parent[path[-1]])
            path.reverse()
            return True, path, best_g[goal], nodes_expanded, time.time() - start_time

        for nb, cost in graph.get(cur, []):
            new_g = best_g[cur] + cost
            if nb not in best_g or new_g <= best_g[nb]:
                best_g[nb] = new_g
                parent[nb] = cur
                heapq.heappush(frontier, (new_g + h[nb], nb))

    return False, [], 0, nodes_expanded, time.time() - start_time


def print_result(algo_name, found, path, cost, nodes_expanded, exec_time):
    print(f"Algorithm: {algo_name}")
    if not found:
        print("Path Found: No")
        print(f"Nodes Expanded = {nodes_expanded}")
        print(f"Execution Time = {exec_time:.6f}")
        return
    print("Path Found: Yes")
    print("Path: " + " -> ".join(path))
    print(f"Total Cost = {cost}")
    print(f"Nodes Expanded = {nodes_expanded}")
    print(f"Execution Time = {exec_time:.6f}")


def main():
    graph, h, start, goal = read_input()

    g_found, g_path, g_cost, g_nodes, g_time = greedy_best_first(graph, h, start, goal)
    print_result("Greedy Best-First Search", g_found, g_path, g_cost, g_nodes, g_time)

    print()

    a_found, a_path, a_cost, a_nodes, a_time = a_star(graph, h, start, goal)
    print_result("A* Search", a_found, a_path, a_cost, a_nodes, a_time)

    print()
    print("Comparison:")
    if g_found and a_found:
        print(f"Path cost -> Greedy = {g_cost}, A* = {a_cost}")
    print(f"Nodes expanded -> Greedy = {g_nodes}, A* = {a_nodes}")
    print(f"Execution time -> Greedy = {g_time:.6f}, A* = {a_time:.6f}")
    print("A* is optimal since it accounts for both the cost so far (g) and the")
    print("estimated cost to goal (h). Greedy only looks at h, so it can walk into")
    print("a locally attractive but globally worse route.")


if __name__ == "__main__":
    main()


Algorithm: Greedy Best-First Search
Path Found: Yes
Path: A -> C -> E -> F
Total Cost = 10
Nodes Expanded = 4
Execution Time = 0.000000

Algorithm: A* Search
Path Found: Yes
Path: A -> C -> D -> E -> F
Total Cost = 10
Nodes Expanded = 5
Execution Time = 0.000000

Comparison:
Path cost -> Greedy = 10, A* = 10
Nodes expanded -> Greedy = 4, A* = 5
Execution time -> Greedy = 0.000000, A* = 0.000000
A* is optimal since it accounts for both the cost so far (g) and the
estimated cost to goal (h). Greedy only looks at h, so it can walk into
a locally attractive but globally worse route.
